<a href="https://colab.research.google.com/github/vishaljoshi24/DungeonsAndDragonsTurnClassification/blob/Codex-Branch/prompt_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
print(sys.version)
print(sys.executable)

!{sys.executable} -m pip --version

In [ ]:
!git clone -b Codex-Branch https://github.com/vishaljoshi24/DungeonsAndDragonsTurnClassification/
%cd /content/DungeonsAndDragonsTurnClassification

In [ ]:
import requests

response = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)
print(response.status_code)
print(response.json())

In [ ]:
import subprocess

result = subprocess.run(
    ["ollama", "pull", "qwen3:4b"],
    capture_output=False,
    text=True,
)

print("Download finished with exit code:", result.returncode)

In [ ]:
import requests

response = requests.get("http://127.0.0.1:11434/api/tags", timeout=10)
response.raise_for_status()

models = response.json()["models"]
for model in models:
    print(model["name"])


In [ ]:
!pip install dspy==3.2.1

In [ ]:
import pandas as pd
import dspy

In [ ]:
generation_df = pd.read_excel('Articulation Work Definitions.xlsx')

In [ ]:
generation_df

In [ ]:
# generation_df.drop(columns=['Jack\'s Codes'], inplace=True)

In [ ]:
# generation_df.drop(columns=['Vishal\'s Codes'], inplace=True)

In [ ]:
# generation_df.drop(columns=['Notes'], inplace=True)

In [ ]:
generation_df

In [ ]:
genset = []

for category, definition in generation_df.values:
    examples = dspy.Example(context=category, question=definition).with_inputs("context", "question")
    genset.append(examples)

In [ ]:
genset

In [ ]:
lm = dspy.LM('ollama_chat/qwen3:4b', api_base = 'http://localhost:11434', api_key='', max_tokens=8192, cache=False)
dspy.configure(lm=lm)

In [ ]:
from typing import Literal


class TurnClassifier(dspy.Signature):
    """Given the context for a Dungeons & Dragons game turn and the game turn itself, classify the turn."""
    context: str = dspy.InputField(desc = "The three previous game turns which describe a player's action or their dialogue.")
    question: str = dspy.InputField (desc="The current turn taken by a player, which can include a description of an action or a piece of dialogue.")
    response: Literal['question game-state',
                      'assert game-state',
                      'act on game-state',
                      'propose intention or action',
                      'argue',
                      'acknowledge',
                      'probe intention',
                      'interact with resource',
                      'out-of-gameplay',
                      ] = dspy.OutputField()

class PlayerInstruction(dspy.Signature):
  """Given the category of articulation work and its definition, generate a prompt instruction for how to conduct that type of articulation work."""
  context: str = dspy.InputField(desc='Category of articulation work carried out in D&D')
  question:str = dspy.InputField(desc='Definition of the type of articulation work.')
  player_instruction:str = dspy.OutputField(desc="instruction on how to behave within a D&D game based on the definition of the type of articulation work.")

In [ ]:
class PromptGenerator(dspy.Module):
  def __init__(self):
    # self.classifier = classify
    self.generator = dspy.ChainOfThought(PlayerInstruction, caching=False)

  def forward(self, context, question, **kwargs):
    # pred_category = classify(context=context, question=question).response
    prompt = self.generator(
        context=context,
        question=question
    )
    return prompt

prompt_generator = PromptGenerator()


In [ ]:
prompt_generator(
    context=genset[0]['context'],
    question=genset[0]['question']
)

In [ ]:
generated_prompt = []

for i in range(len(genset)):
    generated_prompt.append(prompt_generator(genset[i]['context'], genset[i]['question']))

In [ ]:
generated_prompt

In [ ]:
generated_prompt[0]['player_instruction']

In [ ]:
genset[0]

In [ ]:
player_instruction = []
articulation_work=[]

for i in range(len(generated_prompt)):
  articulation_work.append(genset[i]['context'])
  player_instruction.append(generated_prompt[i])


In [ ]:
articulation_work

In [ ]:
player_instruction

In [ ]:
articulation_prompts = {
    category: prediction.player_instruction
    for category, prediction in zip(articulation_work, player_instruction)
}

In [ ]:
articulation_prompts

In [ ]:
instruction_df.drop_duplicates(subset=[0], inplace=False)

In [ ]:
filename = 'instructions.xlsx'
instruction_df.to_excel(filename)